# 09 — Class Decorators
**Advanced Python · 03_Decorators · Notebook 09/10**

This notebook covers two distinct ideas: **class-based decorators** (classes that decorate functions) and **class decorators** (functions that decorate classes). It builds on the previous notebook about forwarding function arguments and preserving return values.

## 1. Introduction

A **class-based decorator** is an object that wraps a function, usually using `__init__()` and `__call__()`. A **class decorator** receives a class and returns that class or a replacement.

| Class-based decorator | Class decorator |
|---|---|
| A class decorates a function | A function decorates a class |
| Often stores a function on an instance | Receives the class object |
| Often uses `__call__()` | Usually returns the modified class |

```text
Function → decorator class instance → callable wrapper
Class → decorator function → modified class
```

## 2. Function-Based vs Class-Based Decorators

A function-based decorator returns an inner wrapper. A class-based decorator can instead return an instance whose `__call__()` method performs the wrapping.

In [ ]:
from functools import wraps, update_wrapper

def before_call(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print("Before")
        return func(*args, **kwargs)
    return wrapper

@before_call
def hello():
    return "Hello!"

print(hello())

## 3. Review of Callable Objects

Functions are callable objects. An instance of a class can also be callable if the class defines `__call__()`.

In [ ]:
def add(a, b):
    return a + b

print(callable(add))
print(callable(42))

## 4. Introduction to `__call__()`

When an instance defines `__call__()`, `instance(...)` invokes that method. This is what lets a decorator object behave like a function.

## 5. Creating a Callable Class

In [ ]:
class Greeter:
    def __call__(self, name):
        return f"Hello, {name}!"

greet = Greeter()
print(greet("Komal"))
print(callable(greet))

## 6. Creating a Class-Based Decorator

The constructor stores the original function. The callable instance surrounds its execution and returns its result.

In [ ]:
class MyDecorator:
    def __init__(self, func):
        self.func = func
        update_wrapper(self, func)

    def __call__(self):
        print("Before function")
        result = self.func()
        print("After function")
        return result

@MyDecorator
def greet():
    print("Hello!")

greet()

## 7. How a Class-Based Decorator Works

`@MyDecorator` is equivalent to `greet = MyDecorator(greet)`. At decoration time, Python constructs an instance and stores the original function in `self.func`. At call time, `greet()` invokes the instance's `__call__()` method.

```text
Original greet function
        ↓  MyDecorator(greet)
Decorator instance stores function
        ↓  greet()
       __call__()
        ↓
Original function runs
```

In [ ]:
print(isinstance(greet, MyDecorator))
print(greet.__name__)  # update_wrapper preserved this metadata

## 8. Class-Based Decorators with Function Arguments

As in `08_Decorators_with_Function_Arguments.ipynb`, the wrapper forwards positional and keyword arguments and preserves the return value. Here, the wrapper is `__call__()`.

In [ ]:
class LogArguments:
    def __init__(self, func):
        self.func = func
        update_wrapper(self, func)

    def __call__(self, *args, **kwargs):
        print("Positional:", args)
        print("Keyword:", kwargs)
        result = self.func(*args, **kwargs)
        return result

@LogArguments
def introduce(name, age, city="Hyderabad"):
    return f"{name}, {age}, {city}"

print(introduce("Komal", age=22))

## 9. Class-Based Decorators with Configuration Arguments

`@Repeat(3)` first constructs `Repeat(3)`. Python then calls that instance with the original function. In this pattern, `__call__()` returns an inner wrapper, rather than executing the original function immediately.

In [ ]:
class Repeat:
    def __init__(self, times):
        if times < 1:
            raise ValueError("times must be at least 1")
        self.times = times

    def __call__(self, func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            result = None
            for _ in range(self.times):
                result = func(*args, **kwargs)
            return result
        return wrapper

@Repeat(3)
def greet(name):
    print(f"Hello, {name}")

greet("Komal")

Compare the two forms:

- `@MyDecorator`: `__init__(func)` receives the function; the instance's `__call__()` runs when the decorated function is called.
- `@Repeat(3)`: `__init__(3)` receives configuration; the instance's `__call__(func)` receives the function and returns a wrapper.

## 10. Maintaining State with Class-Based Decorators

Instance attributes can store state across calls. Each decorated function normally receives its own decorator instance.

In [ ]:
class CountCalls:
    def __init__(self, func):
        self.func = func
        self.count = 0
        update_wrapper(self, func)

    def __call__(self, *args, **kwargs):
        self.count += 1
        print(f"Call number: {self.count}")
        return self.func(*args, **kwargs)

@CountCalls
def greet():
    print("Hello!")

greet()
greet()
print("Total calls:", greet.count)

In [ ]:
@CountCalls
def goodbye():
    print("Goodbye!")

goodbye()
print("greet:", greet.count, "goodbye:", goodbye.count)

## 11. Introduction to Decorating Classes

Now the **decorated object is a class**, not a function. A class decorator takes a class object, optionally changes it, and returns the class (or a replacement).

## 12. Creating a Class Decorator

In [ ]:
def add_category(cls):
    cls.category = "Student"
    return cls

@add_category
class Student:
    pass

print(Student.category)

`@add_category` above is equivalent to `Student = add_category(Student)`. Here the function decorates the **class** `Student`.

## 13. Modifying a Class Using a Decorator

A class decorator can attach metadata or modify behavior. Prefer clear, limited changes; unexpected mutations make classes harder to understand.

In [ ]:
def add_version(cls):
    cls.version = "1.0"
    return cls

@add_version
class Report:
    def title(self):
        return "Monthly report"

report = Report()
print(report.title())
print(Report.version)

## 14. Adding Attributes and Methods to Classes

The next example adds a `display_info()` method. It returns the **same class object**, now with an extra method.

In [ ]:
def add_display_info(cls):
    def display_info(self):
        return f"Name: {self.name}, Age: {self.age}"
    cls.display_info = display_info
    return cls

@add_display_info
class Student:
    def __init__(self, name, age):
        self.name = name
        self.age = age

student = Student("Komal", 22)
print(student.display_info())

Class decorators can also replace a class or modify an existing method, but changing behavior dynamically should be deliberate and documented. Avoid overwriting an existing attribute or method by accident.

## 15. Practical Examples

These examples combine the main ideas without introducing new decorator machinery.

### Example A — Call counter

In [ ]:
@CountCalls
def multiply(a, b):
    return a * b

print(multiply(4, 5))
print(multiply(2, 6))
print("Calls:", multiply.count)

### Example B — Configurable repeat

In [ ]:
@Repeat(2)
def announce(message):
    print(message)
    return "Done"

print(announce("Ready"))

### Example C — Execution timer

This reuses the timing pattern from the previous notebook.

In [ ]:
import time

class Timer:
    def __init__(self, func):
        self.func = func
        update_wrapper(self, func)

    def __call__(self, *args, **kwargs):
        start = time.perf_counter()
        try:
            return self.func(*args, **kwargs)
        finally:
            elapsed = time.perf_counter() - start
            print(f"{self.func.__name__}: {elapsed:.6f} seconds")

@Timer
def sum_numbers(limit):
    return sum(range(limit))

print(sum_numbers(100_000))

### Example D — Add class metadata

In [ ]:
def add_metadata(cls):
    cls.category = "Analytics"
    cls.created_by = "Python Learner"
    return cls

@add_metadata
class Dashboard:
    pass

print(Dashboard.category, Dashboard.created_by)

### Example E — Add a class method

In [ ]:
def add_description(cls):
    def describe(self):
        return f"This is a {type(self).__name__} instance."
    cls.describe = describe
    return cls

@add_description
class Project:
    pass

print(Project().describe())

## 16. Function-Based vs Class-Based Decorators

| Aspect | Function-based | Class-based |
|---|---|---|
| Structure | Nested functions | Class with `__call__()` |
| State | Closures or other storage | Instance attributes |
| Configuration | Decorator factory | Constructor or other class methods |
| Common use | Simple wrappers | Stateful or object-oriented wrappers |

Either approach can accomplish many of the same tasks. Choose based on the behavior and state you need to manage.

## 17. Common Mistakes

1. **Confusing the two meanings:** a class-based decorator decorates a function; a class decorator decorates a class.
2. **Forgetting `__call__()`:** an instance without it cannot be used as a callable wrapper.
3. **Not forwarding arguments:** use `self.func(*args, **kwargs)` when appropriate.
4. **Dropping return values:** return the original function's result.
5. **Accidentally sharing state:** use a separate decorator instance per function unless shared state is intentional.
6. **Assuming metadata is preserved:** class-based decorators do not automatically retain `__name__`, `__doc__`, etc. As covered in `07_Functools_Wraps.ipynb`, `functools.update_wrapper(self, func)` can copy important metadata onto a callable instance.

In [ ]:
# Observe that each decorated function has its own counter.
@CountCalls
def first():
    return "first"

@CountCalls
def second():
    return "second"

first()
first()
second()
print(first.count, second.count)

## 18. Summary

- **Class-based decorator:** a class wraps a function; `__init__()` stores the function and `__call__()` executes it.
- **Configurable class-based decorator:** constructor receives configuration and `__call__()` receives the function and returns a wrapper.
- **Stateful decorator:** instance attributes track state across calls.
- **Class decorator:** a function receives and returns a class, optionally adding attributes or methods.
- Forward arguments, preserve return values, and consider metadata preservation.

### Practice

1. Create a class-based decorator that counts function calls.
2. Build a configurable decorator that prints a custom message before execution.
3. Write a class decorator that adds a `created_by` attribute.

**Next:** `10_Practical_Decorator_Patterns.ipynb`